# Banking Form Field Detector

**Hybrid LayoutLMv3 + YOLOv8 pipeline** for detecting question-answer field pairs in scanned form images.

| Component | Role | Model |
|---|---|---|
| LayoutLMv3 | Detects **question label** tokens via OCR + layout | Fine-tuned on your annotated form images |
| YOLOv8n | Detects **answer box** regions visually | Fine-tuned on the same images (answer-only) |

**Inference flow:** LayoutLMv3 finds question text regions → YOLOv8 finds blank answer boxes → spatial cost matching pairs each question to its answer field.

---

### Prerequisites
- Annotate your form images in **Label Studio** using three rectangle labels: `Question`, `Answer`, `Other`
- Export annotations as **JSON** (Label Studio format)
- Install dependencies: `pip install transformers ultralytics torch torchvision Pillow pytesseract scikit-learn tqdm`
- Install **Tesseract OCR**: https://github.com/tesseract-ocr/tesseract (Windows: set path in config cell below)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — update these values before running
# ═══════════════════════════════════════════════════════════════════════════

# Path to your Label Studio JSON export
ANNOTATIONS_JSON = "LMV3_Labels.json"

# Directory containing your form images (JPEG or PNG)
IMAGE_DIR = "form images"

# Path to a test image for inference (Section 4)
TEST_IMAGE = "form images/your_test_form.jpeg"

# Tesseract executable path
#   Windows: set the path below (default install location shown)
#   Linux / macOS: Tesseract is usually on PATH — set to None
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# ── Training hyperparameters ───────────────────────────────────────────────
YOLO_EPOCHS  = 100   # YOLO answer-box detector
LMV3_EPOCHS  = 20    # LayoutLMv3 question detector
BATCH_SIZE   = 2     # reduce to 1 if GPU memory is limited
MAX_LEN      = 512   # LayoutLMv3 max token length

# ── Output paths (no need to change) ──────────────────────────────────────
YOLO_WEIGHTS     = "runs/detect/yolo_answer_detector/weights/best.pt"
LMV3_OUTPUT_DIR  = "./layoutlmv3-question-detector"

## 1. Data Preparation

Convert Label Studio annotations (`LMV3_Labels.json`) to YOLO format, keeping only **Answer** boxes.  
Run `convert_to_yolo_answers.py` once to regenerate `yolo_answer_labels/` if needed.

In [ ]:
import json, os, re, shutil, random
from pathlib import Path

# ── Load annotations ──────────────────────────────────────────────────────
with open(ANNOTATIONS_JSON, encoding="utf-8") as f:
    label_studio_data = json.load(f)
print(f"Loaded {len(label_studio_data)} annotated images from {ANNOTATIONS_JSON}")

# ── 1. Generate answer-only YOLO label files ──────────────────────────────
os.makedirs("yolo_answer_labels", exist_ok=True)

for task in label_studio_data:
    file_upload = task.get("file_upload", "")
    image_name  = re.sub(r"^[0-9a-f]{8}-", "", file_upload)  # strip Label Studio UUID prefix
    stem        = Path(image_name).stem
    lines       = []
    for ann in task.get("annotations", []):
        for result in ann.get("result", []):
            if result.get("type") != "rectanglelabels":
                continue
            if "Answer" not in result.get("value", {}).get("rectanglelabels", []):
                continue
            v  = result["value"]
            cx = max(0.0, min(1.0, (v["x"] + v["width"]  / 2) / 100))
            cy = max(0.0, min(1.0, (v["y"] + v["height"] / 2) / 100))
            w  = max(0.0, min(1.0, v["width"]  / 100))
            h  = max(0.0, min(1.0, v["height"] / 100))
            lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")  # class 0 = Answer
    with open(f"yolo_answer_labels/{stem}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

print(f"Generated {len(label_studio_data)} label files  →  yolo_answer_labels/")

# ── 2. Build yolo_dataset/ with 80/20 train/val split ────────────────────
image_dir   = Path(IMAGE_DIR)
image_files = sorted(image_dir.glob("*.jpeg")) + sorted(image_dir.glob("*.png"))
random.seed(42)
random.shuffle(image_files)

n_train  = int(0.8 * len(image_files))
splits   = {"train": image_files[:n_train], "val": image_files[n_train:]}

for split_name, files in splits.items():
    (Path("yolo_dataset/images") / split_name).mkdir(parents=True, exist_ok=True)
    (Path("yolo_dataset/labels") / split_name).mkdir(parents=True, exist_ok=True)
    for src in files:
        shutil.copy(src, Path("yolo_dataset/images") / split_name / src.name)
        lbl = Path("yolo_answer_labels") / (src.stem + ".txt")
        dst = Path("yolo_dataset/labels") / split_name / (src.stem + ".txt")
        if lbl.exists():
            shutil.copy(lbl, dst)
        else:
            dst.touch()  # empty label file for images with no answer boxes

print(f"Train: {len(splits['train'])} images  |  Val: {len(splits['val'])} images")

In [ ]:
yaml_content = """\
path: yolo_dataset
train: images/train
val: images/val

nc: 1
names: ['Answer']
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_content)
print("dataset.yaml written  (nc=1, Answer class only)")

## 2. YOLO Answer Box Detector

Fine-tune YOLOv8n to detect blank **answer field** regions visually (underlines, boxes, shaded areas).  
Trains for `YOLO_EPOCHS` epochs (default 100) on your annotated form images — **Answer** class only (class 0).  
Best weights are saved to `runs/detect/yolo_answer_detector/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

# yolov8n.pt is downloaded automatically on first run (~6 MB)
yolo_model = YOLO("yolov8n.pt")

yolo_model.train(
    data="dataset.yaml",
    epochs=YOLO_EPOCHS,
    imgsz=640,
    batch=4,
    project="runs/detect",
    name="yolo_answer_detector",
)

YOLO_WEIGHTS = "runs/detect/yolo_answer_detector/weights/best.pt"
print(f"Training complete. Best weights saved to: {YOLO_WEIGHTS}")

## 3. LayoutLMv3 Question Detector

Fine-tune `microsoft/layoutlmv3-base` on your annotated form images for **2-class token classification**: `Question` vs `Other`.  
Answer tokens are ignored during training — answer fields are blank so OCR finds no text in them; YOLO handles those.  
Model saved to the path set in `LMV3_OUTPUT_DIR` (default: `layoutlmv3-question-detector/`).

In [ ]:
import json
from pathlib import Path
from PIL import Image
from collections import Counter
import random
import torch
from torch.utils.data import DataLoader
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from torch.optim import AdamW
from tqdm import tqdm
import pytesseract

# ── Tesseract setup ───────────────────────────────────────────────────────
# On Linux/macOS Tesseract is usually on PATH — leave TESSERACT_CMD as None.
# On Windows set TESSERACT_CMD in the configuration cell at the top.
if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

lmv3_processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=True)

# 2-class: LayoutLMv3 distinguishes Question vs Other.
# Answer fields are blank in forms → no OCR text → YOLO handles them visually.
LABEL_MAP  = {"Other": 0, "Question": 1, "Answer": -1}  # Answer → -1 (ignored in loss)
ID2LABEL   = {0: "Other", 1: "Question"}
NUM_LABELS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def iow(label_box, word_box):
    """Intersection-over-Word-area: fraction of the word bbox covered by the label box."""
    xA = max(label_box[0], word_box[0])
    yA = max(label_box[1], word_box[1])
    xB = min(label_box[2], word_box[2])
    yB = min(label_box[3], word_box[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    word_area = (word_box[2] - word_box[0]) * (word_box[3] - word_box[1])
    return inter / word_area if word_area > 0 else 0

image_dir   = Path(IMAGE_DIR)
_img_lookup = {p.stem.lower().replace(" ", "_"): p
               for p in list(image_dir.glob("*.jpeg")) + list(image_dir.glob("*.png"))}

def resolve_image(file_upload_name):
    """Map a Label Studio file_upload name to a local image path."""
    try:
        filename = file_upload_name.split("-", 1)[1]   # strip UUID prefix
    except IndexError:
        filename = file_upload_name
    key = Path(filename).stem.lower().replace(" ", "_")
    if key in _img_lookup:
        return _img_lookup[key]
    p = image_dir / filename
    return p if p.exists() else None

def process_image_item(img, raw_annotations):
    """
    OCR-encode one image and assign per-token labels:
      Question (1) or Other (0).  Answer tokens → -100 (ignored in loss).
    First subword of each OCR word gets the label; subsequent subwords → -100.
    """
    label_boxes = []
    for ann in raw_annotations:
        v        = ann["value"]
        lbl_name = v["rectanglelabels"][0]
        if lbl_name == "Answer":
            continue  # blank fields have no OCR text — YOLO detects these
        lbl = LABEL_MAP[lbl_name]
        x1  = int((v["x"]               / 100) * 1000)
        y1  = int((v["y"]               / 100) * 1000)
        x2  = int(((v["x"] + v["width"])  / 100) * 1000)
        y2  = int(((v["y"] + v["height"]) / 100) * 1000)
        label_boxes.append((x1, y1, x2, y2, lbl))

    enc = lmv3_processor(
        img, return_tensors="pt",
        padding="max_length", max_length=MAX_LEN, truncation=True,
    )
    word_ids   = enc.word_ids(batch_index=0)
    tok_bboxes = enc["bbox"][0].tolist()

    word_bbox = {}
    for tok_i, wid in enumerate(word_ids):
        if wid is not None and wid not in word_bbox:
            word_bbox[wid] = tok_bboxes[tok_i]

    word_label = {}
    for wid, wb in word_bbox.items():
        best_lbl, best_score = 0, 0.0
        for (lx1, ly1, lx2, ly2, lbl) in label_boxes:
            score = iow((lx1, ly1, lx2, ly2), wb)
            if score > best_score:
                best_score, best_lbl = score, lbl
        word_label[wid] = best_lbl

    seen, token_labels = set(), []
    for wid in word_ids:
        if wid is None:
            token_labels.append(-100)
        elif wid not in seen:
            seen.add(wid)
            token_labels.append(word_label.get(wid, 0))
        else:
            token_labels.append(-100)

    return {
        "input_ids":      enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0],
        "bbox":           enc["bbox"][0],
        "pixel_values":   enc["pixel_values"][0],
        "labels":         torch.tensor(token_labels, dtype=torch.long),
    }

In [ ]:
# Process all annotated images into LayoutLMv3 samples
with open(ANNOTATIONS_JSON, encoding="utf-8") as f:
    lmv3_raw = json.load(f)

all_samples = []
for item in lmv3_raw:
    img_path = resolve_image(item["file_upload"])
    if img_path is None:
        print(f"  [SKIP] image not found for: {item['file_upload']}")
        continue
    img  = Image.open(img_path).convert("RGB")
    anns = [a for a in item["annotations"][0]["result"] if a["type"] == "rectanglelabels"]
    all_samples.append(process_image_item(img, anns))
    print(f"  [OK] {img_path.name}")

print(f"\nProcessed {len(all_samples)} / {len(lmv3_raw)} images")

counts = Counter()
for s in all_samples:
    for l in s["labels"].tolist():
        if l != -100:
            counts[l] += 1
print("Label distribution:", {ID2LABEL[k]: v for k, v in sorted(counts.items())})

In [ ]:
# Prepare DataLoaders with 80/20 split

random.seed(42)
idx = list(range(len(all_samples)))
random.shuffle(idx)

split      = int(0.8 * len(all_samples))   # 80 / 20
train_data = [all_samples[i] for i in idx[:split]]
val_data   = [all_samples[i] for i in idx[split:]]
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

def lmv3_collate(batch):
    return {k: torch.stack([s[k] for s in batch]) for k in batch[0]}

train_loader = DataLoader(train_data, batch_size=2, shuffle=True,  collate_fn=lmv3_collate)
val_loader   = DataLoader(val_data,   batch_size=2, shuffle=False, collate_fn=lmv3_collate)

# Sanity check
b = next(iter(train_loader))
print("input_ids   :", b['input_ids'].shape)
print("pixel_values:", b['pixel_values'].shape)

print("labels      :", b['labels'].shape)

In [ ]:
# Load model and move to device

lmv3_model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL_MAP,
)
lmv3_model.to(device)
print("Model loaded on", device)

In [ ]:
# Train with AdamW optimizer and gradient clipping

optimizer = AdamW(lmv3_model.parameters(), lr=5e-5, weight_decay=0.01)

for epoch in range(LMV3_EPOCHS):
    lmv3_model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{LMV3_EPOCHS}"):
        optimizer.zero_grad()
        out = lmv3_model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            bbox           = batch['bbox'].to(device),
            pixel_values   = batch['pixel_values'].to(device),
            labels         = batch['labels'].to(device),
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(lmv3_model.parameters(), 1.0)
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1}/{LMV3_EPOCHS}  Train Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
from sklearn.metrics import classification_report

lmv3_model.eval()
flat_preds, flat_labels = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        out = lmv3_model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            bbox           = batch['bbox'].to(device),
            pixel_values   = batch['pixel_values'].to(device),
        )
        preds  = out.logits.argmax(dim=-1).cpu().numpy()
        labels = batch['labels'].cpu().numpy()
        for p_seq, l_seq in zip(preds, labels):
            for p, l in zip(p_seq, l_seq):
                if l != -100:
                    flat_preds.append(ID2LABEL[p])
                    flat_labels.append(ID2LABEL[l])

print(classification_report(flat_labels, flat_preds, labels=["Question", "Other"]))

In [ ]:
lmv3_model.save_pretrained(LMV3_OUTPUT_DIR)
lmv3_processor.save_pretrained(LMV3_OUTPUT_DIR)
print(f"Model saved to {LMV3_OUTPUT_DIR}")

## 4. Inference — Q→A Pair Detection

This section standalone (skip training above) tests on a new form image using the saved models.

**Pipeline:**  
1. LayoutLMv3 → token-level `Question` predictions → merge into field-label bounding boxes  
2. YOLOv8 → detect blank answer boxes  
3. Spatial cost matching → pair each question label to its nearest answer field

In [ ]:
# ── Load saved models ─────────────────────────────────────────────────────
# Run this cell to use pre-trained models without re-running Sections 2 & 3.
import torch
from PIL import Image
from ultralytics import YOLO
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import pytesseract

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ID2LABEL = {0: "Other", 1: "Question"}

# YOLO answer-box detector
yolo_model = YOLO(YOLO_WEIGHTS)

# LayoutLMv3 question detector
lmv3_processor = LayoutLMv3Processor.from_pretrained(LMV3_OUTPUT_DIR)
lmv3_model     = LayoutLMv3ForTokenClassification.from_pretrained(LMV3_OUTPUT_DIR)
lmv3_model.to(device).eval()

print(f"Models loaded on {device}")
print(f"  YOLO:     {YOLO_WEIGHTS}")
print(f"  LMv3:     {LMV3_OUTPUT_DIR}")

In [ ]:
# Run LayoutLMv3 on the test image
test_img = Image.open(TEST_IMAGE).convert("RGB")
test_enc = lmv3_processor(
    test_img, return_tensors="pt",
    padding="max_length", max_length=MAX_LEN, truncation=True,
)
test_enc = {k: v.to(device) for k, v in test_enc.items()}

with torch.no_grad():
    out = lmv3_model(**test_enc)

preds        = out.logits.argmax(dim=-1)[0]
token_labels = [ID2LABEL[p.item()] for p in preds]
print(f"Predicted tokens — Question: {token_labels.count('Question')}, Other: {token_labels.count('Other')}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

W, H = test_img.size
bboxes_arr    = test_enc["bbox"][0].cpu().numpy()
tok_preds_arr = preds.cpu().numpy()

# ── Collect unique Question word-boxes ────────────────────────────────────
q_boxes_raw = sorted(set(
    tuple(int(v) for v in bbox)
    for bbox, pred in zip(bboxes_arr, tok_preds_arr)
    if ID2LABEL[pred] == "Question" and tuple(int(v) for v in bbox) != (0, 0, 0, 0)
), key=lambda b: (b[1], b[0]))

# ── Merge adjacent word-boxes on the same row into field-label regions ────
GAP_THRESH  = 30   # max horizontal gap (0-1000 units) to merge
MAX_MERGE_W = 250  # merged region max width (~25% of page)

def merge_adjacent(boxes, gap=GAP_THRESH, max_w=MAX_MERGE_W):
    boxes  = [list(b) for b in boxes]
    used   = [False] * len(boxes)
    merged = []
    for i in range(len(boxes)):
        if used[i]:
            continue
        g, used[i] = boxes[i][:], True
        changed = True
        while changed:
            changed = False
            for j in range(len(boxes)):
                if used[j]:
                    continue
                bj = boxes[j]
                if (min(g[3], bj[3]) - max(g[1], bj[1]) > 0
                        and max(bj[0]-g[2], g[0]-bj[2], 0) <= gap
                        and max(g[2], bj[2]) - min(g[0], bj[0]) <= max_w):
                    g[0], g[1] = min(g[0], bj[0]), min(g[1], bj[1])
                    g[2], g[3] = max(g[2], bj[2]), max(g[3], bj[3])
                    used[j] = True
                    changed = True
        merged.append(tuple(g))
    return merged

merged_q_boxes = merge_adjacent(q_boxes_raw)
print(f"Word-level boxes: {len(q_boxes_raw)}  →  Merged question regions: {len(merged_q_boxes)}")

# ── Visualize question regions ────────────────────────────────────────────
fig, ax = plt.subplots(1, figsize=(14, 18))
ax.imshow(test_img)
for i, (x1, y1, x2, y2) in enumerate(merged_q_boxes):
    px, py = x1/1000*W, y1/1000*H
    pw, ph = (x2-x1)/1000*W, (y2-y1)/1000*H
    ax.add_patch(patches.Rectangle((px, py), pw, ph,
                 linewidth=2, edgecolor="blue", facecolor="blue", alpha=0.25))
    ax.text(px, py-4, f"Q{i+1}", fontsize=8, color="blue", fontweight="bold", va="bottom")
ax.set_title(f"LayoutLMv3  |  {len(merged_q_boxes)} Question regions detected", fontsize=13)
ax.axis("off")
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display
from pathlib import Path

def _cx(b): return (b[0]+b[2])/2
def _cy(b): return (b[1]+b[3])/2

# ── 1. Run YOLO to detect Answer boxes ────────────────────────────────────
yolo_results   = yolo_model.predict(test_img, conf=0.25, verbose=False)
yolo_boxes     = yolo_results[0].boxes
ans_boxes_1000 = []
if yolo_boxes is not None and len(yolo_boxes):
    for (bx1, by1, bx2, by2) in yolo_boxes.xyxy.cpu().numpy():
        ans_boxes_1000.append((
            int(bx1/W*1000), int(by1/H*1000),
            int(bx2/W*1000), int(by2/H*1000),
        ))
print(f"YOLO answer boxes: {len(ans_boxes_1000)}")

# ── 2. Filter question boxes whose centre falls inside an answer box ───────
def center_inside(q, a):
    return a[0] <= _cx(q) <= a[2] and a[1] <= _cy(q) <= a[3]

real_q_boxes = [q for q in merged_q_boxes
                if not any(center_inside(q, a) for a in ans_boxes_1000)]
print(f"Question regions after filtering: {len(real_q_boxes)}")

# ── 3. Spatial cost matching — each question → nearest answer box ──────────
MAX_COST = 400   # unmatched if nearest answer is further than this

def qa_cost(q, a):
    dx        = max(0, max(q[0], a[0]) - min(q[2], a[2]))
    dy        = max(0, max(q[1], a[1]) - min(q[3], a[3]))
    edge_dist = (dx**2 + dy**2) ** 0.5
    penalty   = 1.0
    if _cy(a) - _cy(q) < -50:  penalty *= 3.0   # answer above question
    if _cx(a) - _cx(q) < -100: penalty *= 2.0   # answer left of question
    return edge_dist * penalty

qa_pairs = []
for q in real_q_boxes:
    if not ans_boxes_1000:
        break
    costs     = [qa_cost(q, a) for a in ans_boxes_1000]
    best_i    = int(np.argmin(costs))
    best_cost = costs[best_i]
    if best_cost <= MAX_COST:
        qa_pairs.append((q, ans_boxes_1000[best_i]))
    else:
        print(f"  Q unmatched — nearest cost {best_cost:.0f} > MAX_COST {MAX_COST}")
print(f"Q→A pairs matched: {len(qa_pairs)}")

# ── 4. Visualize ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, figsize=(14, 18))
ax.imshow(test_img)

for i, (q, a) in enumerate(qa_pairs):
    qx, qy = q[0]/1000*W, q[1]/1000*H
    qw, qh = (q[2]-q[0])/1000*W, (q[3]-q[1])/1000*H
    ax.add_patch(patches.Rectangle((qx, qy), qw, qh,
                 linewidth=2, edgecolor="blue", facecolor="blue", alpha=0.2))
    ax.text(qx, qy-4, f"Q{i+1}", fontsize=8, color="blue",
            fontweight="bold", va="bottom")

    ax2, ay2 = a[0]/1000*W, a[1]/1000*H
    aw, ah   = (a[2]-a[0])/1000*W, (a[3]-a[1])/1000*H
    ax.add_patch(patches.Rectangle((ax2, ay2), aw, ah,
                 linewidth=2, edgecolor="orange", facecolor="orange", alpha=0.2))
    ax.text(ax2+aw, ay2-4, f"A{i+1}", fontsize=8, color="darkorange",
            fontweight="bold", va="bottom", ha="right")

    ax.annotate("", xy=(_cx(a)/1000*W, _cy(a)/1000*H),
                    xytext=(_cx(q)/1000*W, _cy(q)/1000*H),
                arrowprops=dict(arrowstyle="-|>", color="green", lw=1.5, mutation_scale=12))

ax.set_title(
    f"Q→A Pairs  |  {len(qa_pairs)} matched  (image: {Path(TEST_IMAGE).name})",
    fontsize=13,
)
ax.axis("off")
plt.tight_layout()
display(fig)
plt.close(fig)